In [2]:
import sys
sys.path.append('..')

In [70]:
import tqdm
import torch
import pickle
import warnings
import numpy as np
import pandas as pd
import torch.optim as optim
from copy import deepcopy
import plotly.express as px
import plotly.graph_objects as go
from sklearn.model_selection import ParameterGrid

from src.data import *
from src.utils import *
from src.model import *
from src.recourse import *

warnings.filterwarnings('ignore')

In [50]:
# file_path = "../results/cost_validity/lr_synthetic_alg1_lamb0.1.pickle"
# ret = pd.read_pickle(file_path)

# x0 = ret['x_0'][1][0]
# x0_withBias = np.hstack((x0, np.array([1])))
# xR_old = ret['x_r'][1][0]
# xR_old_withBias = np.hstack((xR_old, np.array([1])))

# theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64)
# bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64) 
# divider = np.linalg.norm(theta0)
# theta0 = ret['theta_0'][0][0][0][0].numpy().astype(np.float64) / divider
# bias0 = ret['theta_0'][0][0][1].numpy().astype(np.float64) /divider
# theta0_withBias = np.hstack((theta0, bias0))

# alpha = 0.02


In [ ]:
alphas = (0.02, 0.04, 0.2)      # <------------- Don't Touch
clf = "lr"
datasets = ["synthesis"]
recourse_fns = [LARRecourse]    # <---------------------
lambdas = [0.1, 0.2, 0.3]       # <-------------
save_file = False



for dataset in datasets:
    # Read File and Get results
    file_path = f"../results/cost_validity_latest/{dataset}_correct.pickle"
    ret = pd.read_pickle(file_path)

    x0s = np.array(ret['x_0'])
    x0s = x0s[0]
    theta0 = ret['theta_0']['out.weight'][0].numpy().astype(np.float64)
    bias0 = ret['theta_0']['out.bias'].numpy().astype(np.float64) 

    ret['x_0'] = ret['x_0'][:len(alphas)]
    filtered_paramVal = [val['delta_max'] for val in ret['params'] if val['delta_max'] in alphas]
    ret['params'] = ParameterGrid({'delta_max': filtered_paramVal})

    for recourse_fn in recourse_fns:
        for lamb in lambdas:
            xRs = [] # Final recouuse list
            
            for alpha in alphas:
                res = []
                for x0 in tqdm.tqdm(x0s, desc=f"Running {clf}_{dataset} with lambda = {lamb}, alpha = {alpha}"):
                    reco = recourse_fn(weights=theta0, bias=bias0, alpha=alpha, lamb=lamb)
                    res.append(reco.get_recourse(x0))
                xRs.append(res)
            
            ret['x_r'] = xRs
            
            # Save each (dataset, recourse function, lambda) file
            if save_file:
                file_path_saved = f"../results/cost_validity_latest/{clf}_{dataset}_{reco.name}_lamb{lamb}_new.pickle"
                with open(file_path_saved, 'wb') as outFile:
                    pickle.dump(ret, outFile)

Running lr_synthesis with lambda = 0.1, alpha = 0.02: 100%|██████████| 100/100 [00:00<00:00, 6605.10it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.04: 100%|██████████| 100/100 [00:00<00:00, 6226.88it/s]
Running lr_synthesis with lambda = 0.1, alpha = 0.2: 100%|██████████| 100/100 [00:00<00:00, 5018.97it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.02: 100%|██████████| 100/100 [00:00<00:00, 5757.45it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.04: 100%|██████████| 100/100 [00:00<00:00, 8070.78it/s]
Running lr_synthesis with lambda = 0.2, alpha = 0.2: 100%|██████████| 100/100 [00:00<00:00, 5440.37it/s]
Running lr_synthesis with lambda = 0.3, alpha = 0.02: 100%|██████████| 100/100 [00:00<00:00, 7207.57it/s]
Running lr_synthesis with lambda = 0.3, alpha = 0.04: 100%|██████████| 100/100 [00:00<00:00, 8185.76it/s]
Running lr_synthesis with lambda = 0.3, alpha = 0.2: 100%|██████████| 100/100 [00:00<00:00, 4830.20it/s]


[[array([2.15167604, 5.56784421]),
  array([-0.13859741,  1.33934887]),
  array([1.26630203, 4.12126695]),
  array([-0.44973171,  0.79325763]),
  array([1.58189636, 4.63690392]),
  array([2.40014011, 5.97379973]),
  array([-0.18373847,  1.26011897]),
  array([0.55261863, 2.52309469]),
  array([2.36764776, 5.92071179]),
  array([2.94054836, 6.85675122]),
  array([1.57881151, 4.6318637 ]),
  array([-0.22277734,  1.19159942]),
  array([1.71237871, 4.85009383]),
  array([2.96019194, 6.88884607]),
  array([0.02469741, 2.0926587 ]),
  array([1.29618258, 4.17008759]),
  array([1.42386469, 4.37870228]),
  array([0.       , 1.9556052]),
  array([2.68096016, 6.43262041]),
  array([1.95917755, 5.2533286 ]),
  array([0.81507656, 3.38402765]),
  array([0.93275123, 3.57629159]),
  array([2.33238296, 5.86309403]),
  array([2.17322308, 5.60304905]),
  array([2.89387398, 6.78049181]),
  array([-0.42386263,  0.83866206]),
  array([0.       , 1.8924809]),
  array([2.66397875, 6.40487516]),
  array([1.475